# BirdCLEF+ 2026 — Training (Kaggle GPU)

Runs entirely on Kaggle. Attach:
1. The competition dataset `birdclef-2026` (auto-mounted at `/kaggle/input/birdclef-2026`).
2. (Optional) a private utility dataset `birdclef2026-code` containing this repo's `src/` and `data_prep/` folders → mounted under `/kaggle/input/birdclef2026-code/`.

Set the kernel to **GPU (T4 x2 or P100)**. Enabling internet is **not** required for training.

Output: `best.pt` + `effnet_b0.int8.onnx` under `/kaggle/working/` → save as a Kaggle Dataset named e.g. `birdclef2026-weights` and attach to the submission notebook.

In [ ]:
# ---- 1. Locate code + data (resilient to Kaggle path variations) -------
import os, sys, shutil, glob
from pathlib import Path

def _first_existing(candidates):
    for c in candidates:
        for hit in glob.glob(c):
            p = Path(hit)
            if p.exists():
                return p
    return None

DATA = _first_existing([
    '/kaggle/input/birdclef-2026',
    '/kaggle/input/competitions/birdclef-2026',
    '/kaggle/input/*birdclef*2026*',
])
assert DATA is not None and (DATA / 'taxonomy.csv').exists(), 'birdclef-2026 not attached'

CODE_ROOT = _first_existing([
    '/kaggle/input/birdclef2026-code',
    '/kaggle/input/datasets/ahmedsherif382/birdclef2026-code',
    '/kaggle/input/*birdclef2026-code*',
])
assert CODE_ROOT is not None, 'birdclef2026-code dataset not attached'

# Code dataset may be flat (src/, data_prep/) or doubly nested (src/src/...).
def _resolve(parent, name, marker):
    for cand in (parent / name / name, parent / name):
        if (cand / marker).exists():
            return cand
    raise FileNotFoundError(f'cannot find {name}/{marker} under {parent}')

SRC_DIR = _resolve(CODE_ROOT, 'src',       'taxonomy.py')
DP_DIR  = _resolve(CODE_ROOT, 'data_prep', 'make_folds.py')

os.environ['BIRDCLEF_DATA'] = str(DATA)

# Copy to /kaggle/working so imports are writable/stable.
WORK = Path('/kaggle/working')
for src_dir, name in ((SRC_DIR, 'src'), (DP_DIR, 'data_prep')):
    dst = WORK / name
    if dst.exists():
        shutil.rmtree(dst)
    shutil.copytree(src_dir, dst)

# Purge any cached imports so we pick up /kaggle/working, not a stale location.
for mod in list(sys.modules):
    if mod == 'src' or mod.startswith('src.') or mod == 'data_prep' or mod.startswith('data_prep.'):
        del sys.modules[mod]
sys.path[:] = [str(WORK)] + [p for p in sys.path if p != str(WORK)]

print('data root:', DATA)
print('code root:', CODE_ROOT)
print('staged code at', WORK)


In [ ]:
# ---- 2. Imports ----------------------------------------------------------
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

from src.config import DEFAULTS, RUNS_DIR
from src.datasets import SpecAugCfg
from src.ogg_dataset import build_on_the_fly
from src.losses import WeightedBCEWithLogits, FocalBCE, compute_pos_weight
from src.metrics import birdclef_roc_auc
from src.models import build_model
from src.taxonomy import class_to_idx, num_classes
from data_prep.make_folds import build_folds

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device, '  nc:', num_classes())

In [ ]:
# ---- 3. Build site-based folds (writes /kaggle/working/folds.csv) -------
FOLDS_CSV = WORK / 'folds.csv'
folds_df = build_folds(n_folds=5, seed=42)
folds_df.to_csv(FOLDS_CSV, index=False)
print(folds_df.groupby('fold').size())
folds_df.head()

In [ ]:
# ---- 4. Config -----------------------------------------------------------
cfg = dict(
    val_fold=0,
    backbone='tf_efficientnet_b0.ns_jft_in1k',
    pretrained=True,
    drop_rate=0.3,           # ↑ from 0.2 — model was undertrained, but heavier aug needs more reg
    batch_size=96,
    num_workers=4,
    prefetch_factor=3,
    epochs=30,               # ↑ from 12 — main reason prev model underperformed
    warmup_epochs=3,         # NEW: linear warmup before cosine
    lr=3e-4,
    min_lr=1e-6,             # NEW: cosine floor
    weight_decay=1e-4,
    grad_clip=1.0,           # NEW
    ema_decay=0.999,         # NEW
    mixup_alpha=0.5,
    bg_mix_prob=0.5,
    pos_weight_cap=50.0,
    loss='bce',
)
# Slightly stronger SpecAugment to match longer training:
spec_aug = SpecAugCfg(time_masks=2, time_mask_max=60, freq_masks=2, freq_mask_max=24, time_shift_max=30)
train_ds, val_ds = build_on_the_fly(val_fold=cfg['val_fold'], folds_csv=FOLDS_CSV,
                                    spec_aug=spec_aug, bg_mix_prob=cfg['bg_mix_prob'])
print('train:', len(train_ds), ' val:', len(val_ds))


In [ ]:
# ---- 5. Model / loss / optim --------------------------------------------
# Rebuild loaders cleanly each run (handles the zombie-worker crash after interrupts).
import gc
for _name in ('train_loader', 'val_loader'):
    if _name in globals():
        try:
            del globals()[_name]
        except Exception:
            pass
gc.collect()

train_loader = DataLoader(train_ds, batch_size=cfg['batch_size'], shuffle=True,
                          num_workers=cfg['num_workers'], pin_memory=True, drop_last=True,
                          persistent_workers=True, prefetch_factor=cfg['prefetch_factor'])
val_loader = DataLoader(val_ds, batch_size=cfg['batch_size']*2, shuffle=False,
                        num_workers=max(2, cfg['num_workers']//2), pin_memory=True,
                        persistent_workers=True, prefetch_factor=cfg['prefetch_factor'])

model = build_model('timm', backbone=cfg['backbone'], num_classes=num_classes(),
                    pretrained=cfg['pretrained'], drop_rate=cfg['drop_rate']).to(device)

# pos_weight from class frequencies in training index.
targets_sum = torch.zeros(num_classes())
c2i = class_to_idx()
for s in train_ds.df['labels'].fillna(''):
    for code in str(s).replace(',', ';').split(';'):
        c = code.strip()
        if c in c2i:
            targets_sum[c2i[c]] += 1
pos_weight = compute_pos_weight(targets_sum, n_samples=len(train_ds), cap=cfg['pos_weight_cap'])
criterion = WeightedBCEWithLogits(pos_weight=pos_weight) if cfg['loss']=='bce' else FocalBCE()

optim = torch.optim.AdamW(model.parameters(), lr=cfg['lr'], weight_decay=cfg['weight_decay'])

# Warmup + cosine: linear ramp from 0 -> lr over warmup_epochs, then cosine to min_lr.
steps_per_epoch = max(1, len(train_loader))
total_steps  = cfg['epochs']        * steps_per_epoch
warmup_steps = cfg['warmup_epochs'] * steps_per_epoch
def _lr_lambda(step):
    if step < warmup_steps:
        return float(step + 1) / float(max(1, warmup_steps))
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    cos = 0.5 * (1.0 + np.cos(np.pi * min(1.0, progress)))
    floor = cfg['min_lr'] / cfg['lr']
    return floor + (1.0 - floor) * cos
sched = torch.optim.lr_scheduler.LambdaLR(optim, _lr_lambda)
scaler = torch.amp.GradScaler('cuda', enabled=torch.cuda.is_available())

# ---- EMA (decay-averaged shadow weights) --------------------------------
class ModelEMA:
    def __init__(self, model, decay=0.999):
        self.decay = decay
        self.shadow = {k: v.detach().clone().float() for k, v in model.state_dict().items()}
    @torch.no_grad()
    def update(self, model):
        d = self.decay
        for k, v in model.state_dict().items():
            if v.dtype.is_floating_point:
                self.shadow[k].mul_(d).add_(v.detach().float(), alpha=1.0 - d)
            else:
                self.shadow[k] = v.detach().clone()
    def apply_to(self, model_copy):
        model_copy.load_state_dict({k: v.to(next(model_copy.parameters()).dtype) for k, v in self.shadow.items()})

ema = ModelEMA(model, decay=cfg['ema_decay'])
# Sibling model used only to evaluate / save EMA weights:
ema_model = build_model('timm', backbone=cfg['backbone'], num_classes=num_classes(),
                        pretrained=False, drop_rate=cfg['drop_rate']).to(device)
print('warmup steps:', warmup_steps, ' total steps:', total_steps, ' EMA decay:', cfg['ema_decay'])


In [ ]:
# ---- 5b. (Optional) Warm-restart from a Kaggle-Dataset model -----------
# Strategy: load weights only, then run a SHORT fine-tune with a LOWER LR
# and a fresh (shorter) cosine schedule. This avoids the "stuck AUC" issue
# caused by fast-forwarding a long cosine schedule into its low-LR tail.
import os, glob

RESUME_MODEL  = None     # set to a path (e.g. '/kaggle/input/birdclef2026-weights/best.pt') to fine-tune
FINETUNE_LR   = 5e-5     # lower than initial 3e-4; warm-restart peak.
FINETUNE_EPS  = 6        # short follow-up run; tune to fit time budget.

start_epoch = 0
best = -1.0
loaded_path = None

# Auto-find best.pt under /kaggle/input if the configured path is wrong.
if RESUME_MODEL and not os.path.exists(RESUME_MODEL):
    hits = sorted(glob.glob('/kaggle/input/**/best.pt', recursive=True))
    if hits:
        print(f'configured path missing; using {hits[0]}')
        RESUME_MODEL = hits[0]
    else:
        print('no best.pt found under /kaggle/input — fresh training')
        RESUME_MODEL = None

if RESUME_MODEL:
    ckpt = torch.load(RESUME_MODEL, map_location=device)
    model.load_state_dict(ckpt['model'])
    best = float(ckpt.get('auc', -1.0))
    loaded_path = RESUME_MODEL
    print(f'loaded weights from {RESUME_MODEL}  (prev best_auc={best:.4f})')

    # Override cfg for the warm-restart phase and rebuild optim + sched fresh.
    cfg['lr']             = FINETUNE_LR
    cfg['epochs']         = FINETUNE_EPS
    cfg['warmup_epochs']  = 1   # short warmup for fine-tune
    optim = torch.optim.AdamW(model.parameters(), lr=cfg['lr'], weight_decay=cfg['weight_decay'])
    steps_per_epoch = max(1, len(train_loader))
    total_steps  = cfg['epochs']        * steps_per_epoch
    warmup_steps = cfg['warmup_epochs'] * steps_per_epoch
    def _lr_lambda(step):
        if step < warmup_steps:
            return float(step + 1) / float(max(1, warmup_steps))
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        cos = 0.5 * (1.0 + np.cos(np.pi * min(1.0, progress)))
        floor = cfg['min_lr'] / cfg['lr']
        return floor + (1.0 - floor) * cos
    sched = torch.optim.lr_scheduler.LambdaLR(optim, _lr_lambda)
    # Reset EMA from the loaded weights so it tracks the warm-restart trajectory:
    ema = ModelEMA(model, decay=cfg['ema_decay'])
    print(f'warm-restart: lr={cfg["lr"]}  epochs={cfg["epochs"]}  '
          f'warmup={cfg["warmup_epochs"]}  total_steps={total_steps}')
else:
    print('fresh training (no checkpoint loaded)')


In [ ]:
# ---- 5c. Sanity-check the save path BEFORE training --------------------
# Writes a tiny dummy ckpt to /kaggle/working/best.pt, reloads it, then
# deletes it. Confirms the directory is writable and torch.save works.
CKPT = WORK / 'best.pt'
print('save target :', CKPT)
print('parent exists:', CKPT.parent.exists(), ' writable:', os.access(CKPT.parent, os.W_OK))

_dummy = {'model': {'_probe': torch.zeros(1)}, 'auc': 0.0, 'epoch': 0, 'cfg': dict(cfg)}
torch.save(_dummy, CKPT)
assert CKPT.exists(), f'torch.save did not create {CKPT}'
_loaded = torch.load(CKPT, map_location='cpu')
assert '_probe' in _loaded['model'], 'reload mismatch'
print(f'OK — wrote/read {CKPT.stat().st_size} bytes at {CKPT}')

# Clean up so cell 6 starts fresh (cell 6 only saves if AUC improves).
CKPT.unlink()
print('probe deleted; ready to train.')


In [ ]:
# ---- 6. Train (resilient: EMA + grad-clip + per-batch try/except + last.pt) ----
import time, traceback
from tqdm.auto import tqdm

def mixup(x, y, alpha):
    if alpha <= 0:
        return x, y
    lam = float(np.random.beta(alpha, alpha))
    idx = torch.randperm(x.size(0), device=x.device)
    return lam * x + (1 - lam) * x[idx], torch.maximum(y, y[idx] * lam)

@torch.no_grad()
def evaluate(eval_model):
    eval_model.eval()
    preds, tgts = [], []
    for b in val_loader:
        try:
            logits = eval_model(b['mel'].to(device, non_blocking=True))
            preds.append(torch.sigmoid(logits).float().cpu().numpy())
            tgts.append(b['target'].numpy())
        except Exception as e:
            print(f'  [eval] skip batch: {e}')
            continue
    if not preds:
        return float('nan')
    return birdclef_roc_auc((np.concatenate(tgts) > 0.5).astype(np.float32),
                            np.concatenate(preds))

CKPT      = WORK / 'best.pt'   # best EMA AUC -> use this for submission
LAST_CKPT = WORK / 'last.pt'   # always overwritten -> safety net if kernel dies
start_epoch = globals().get('start_epoch', 0)
best = globals().get('best', -1.0)

total_t0 = time.time()
for ep in range(start_epoch, cfg['epochs']):
    model.train()
    t0 = time.time()
    n_ok = n_bad = 0
    pbar = tqdm(train_loader, desc=f'ep{ep+1}/{cfg["epochs"]}', leave=False)
    for b in pbar:
        try:
            mel = b['mel'].to(device, non_blocking=True)
            tgt = b['target'].to(device, non_blocking=True)
            wgt = b['weight'].to(device, non_blocking=True)
            mel, tgt = mixup(mel, tgt, cfg['mixup_alpha'])

            optim.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda', enabled=torch.cuda.is_available()):
                logits = model(mel)
                loss = criterion(logits, tgt, wgt)

            if not torch.isfinite(loss):
                n_bad += 1
                continue

            scaler.scale(loss).backward()
            if cfg.get('grad_clip', 0) > 0:
                scaler.unscale_(optim)
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg['grad_clip'])
            scaler.step(optim)
            scaler.update()
            sched.step()
            ema.update(model)

            n_ok += 1
            if n_ok % 50 == 0:
                pbar.set_postfix(loss=float(loss.detach().cpu()),
                                 lr=optim.param_groups[0]['lr'])

        except torch.cuda.OutOfMemoryError:
            print('  [train] OOM - clearing cache and skipping batch')
            n_bad += 1
            torch.cuda.empty_cache()
            continue
        except Exception as e:
            n_bad += 1
            print(f'  [train] skip batch: {type(e).__name__}: {e}')
            continue

    # ---- evaluate EMA weights (smoother, almost always better) ------------
    try:
        ema.apply_to(ema_model)
        ema_auc = evaluate(ema_model)
    except Exception as e:
        print(f'  [eval EMA] failed: {e}')
        traceback.print_exc()
        ema_auc = float('nan')

    try:
        live_auc = evaluate(model)
    except Exception as e:
        print(f'  [eval live] failed: {e}')
        live_auc = float('nan')

    dt = time.time() - t0
    print(f'ep {ep+1}: ema_auc={ema_auc:.4f}  live_auc={live_auc:.4f}  '
          f'ok/bad={n_ok}/{n_bad}  ({dt:.1f}s, total {(time.time()-total_t0)/60:.1f}m)')

    # Always write last.pt (safety net if kernel is killed mid-run).
    try:
        torch.save({'model': ema.shadow, 'cfg': cfg, 'auc': ema_auc, 'epoch': ep+1,
                    'live_state': model.state_dict()}, LAST_CKPT)
    except Exception as e:
        print(f'  [save last] failed: {e}')

    # Save best by EMA AUC (the weights submission notebook will use).
    score = ema_auc if np.isfinite(ema_auc) else live_auc
    if np.isfinite(score) and score > best:
        best = float(score)
        try:
            torch.save({'model': ema.shadow, 'cfg': cfg, 'auc': best, 'epoch': ep+1}, CKPT)
            print(f'  -> saved {CKPT} (ema_auc={best:.4f})')
        except Exception as e:
            print(f'  [save best] failed: {e}')

print(f'done. best ema_auc={best:.4f}  total={(time.time()-total_t0)/60:.1f}m')


In [ ]:
# ---- 7. Export ONNX + INT8 ----------------------------------------------
# Make sure ONNX export deps are present (Kaggle GPU images sometimes lack onnxscript).
import importlib, subprocess, sys, glob
for pkg in ('onnx', 'onnxscript', 'onnxruntime'):
    try:
        importlib.import_module(pkg)
    except ImportError:
        print(f'installing {pkg} ...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

from src.export_onnx import export_cnn, quantize_int8

# Prefer freshly-trained checkpoint in /kaggle/working; fall back to attached model.
LOCAL_CKPT = WORK / 'best.pt'
if LOCAL_CKPT.exists():
    MODEL_CKPT = LOCAL_CKPT
else:
    hits = sorted(glob.glob('/kaggle/input/**/best.pt', recursive=True))
    if not hits:
        raise FileNotFoundError('no best.pt found - run training first or attach the model.')
    MODEL_CKPT = Path(hits[0])
    print(f'no working/best.pt; using {MODEL_CKPT}')

EXPORT_CKPT = MODEL_CKPT
print(f'exporting checkpoint: {EXPORT_CKPT}  ({EXPORT_CKPT.stat().st_size/1e6:.1f} MB)')

FP32 = WORK / f'effnet_b0_fold{cfg["val_fold"]}.onnx'
INT8 = WORK / f'effnet_b0_fold{cfg["val_fold"]}.int8.onnx'
export_cnn(EXPORT_CKPT, FP32, backbone=cfg['backbone'])
quantize_int8(FP32, INT8)
print('fp32:', FP32.stat().st_size/1e6, 'MB')
print('int8:', INT8.stat().st_size/1e6, 'MB')
print('\nNow: File -> Save Version -> Save; then Output -> New Dataset (birdclef2026-weights) and attach to the submission notebook.')
